# 10. Multivariable adjustment

Pathway support can vary alongside cell type, the number of gene annotations, and the strength of the measured response. The audit included these factors together in a regression to ask whether pathway support still tracked prediction accuracy after accounting for them.

**Original agent determination:** The adjusted support coefficient remained positive (about 0.03 per SD after adjustment for DE-gene count).

This notebook reconstructs the June primary run from its original downloads, split files, processing code and saved agent decisions. STATE was evaluated through the official scored-output repositories in that run.

**Portable input setup.** Install `../requirements.txt` in a Python3.11 environment. The first code cell restores the exact offline payloads included in this checkout, verifies their hashes, and then runs the original scored-output analysis. No model inference is performed. Historical saved curves remain visible; a clean run rebuilds its own `computed/` tables. See `../README.md`.

In [1]:
from pathlib import Path
import sys, json
root=next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'recovered_pipeline.py').is_file())
sys.path.insert(0,str(root))
import pandas as pd
from IPython.display import display, Markdown
from state_inputs import ensure_inputs
print(ensure_inputs())
import recovered_pipeline as rp
import notebook_support as ns


{'profile': 'default', 'verified_files': 226, 'assets': [{'asset': 'state-historical-comparison-tables.tar.gz', 'members': 25, 'source': 'verified_existing_inputs'}, {'asset': 'state-official-score-cache.tar.gz', 'members': 193, 'source': 'verified_existing_inputs'}, {'asset': 'state-historical-annotations.tar.gz', 'members': 8, 'source': 'verified_existing_inputs'}], 'new_model_inference': False}


Matplotlib is building the font cache; this may take a moment.


## Original acquisition and inputs

The original downloader and processing scripts are preserved unchanged under `original/`. The replay resolves their downloads to the recovered cache at the recorded repository revisions, so no current dataset replaces the original bytes. Original downloads, processing and plotting are separate visible steps below.

In [2]:
verification=rp.run_stage('controls')
ns.show_acquisition('controls')


,repo,revision
0,arcinstitute/ST-HVG-Tahoe,ca6b751972493f8448e3256d1340ae70ad43e1e7
1,arcinstitute/ST-SE-Tahoe,03b1971d7cc93a7535fd2e957c6948dba267378b
2,arcinstitute/ST-HVG-Replogle,bb6a9562cbbf1fd152df14cc53b4cc7517c77175
3,arcinstitute/ST-SE-Replogle,e324967ff4cea5ec199e29bcbb5c1f00e5b9d69c
4,arcinstitute/ST-HVG-Parse,827df657cd87063026d2abefd73e99036755ac1b
5,arcinstitute/ST-SE-Parse,fa458ffda8690c5c5bdff080dfdc46b46713043b
6,tahoebio/Tahoe-100M,2dc57900b7981cfcf5e211527169a0b006546a95


Original evaluation used official scored CSVs. The replay loads the original cached bytes at these revisions.
All 55 score filenames and the three metadata downloads are in computed/initial/artifacts/download_revisions.json.
Initial acquisition implementation: original/state_generalizability_20260602_single_controller/scripts/run_state_spectra_audit.py
Annotation snapshots and split inputs: original/state_generalizability_continued_20260603/external and original/state_generalizability_20260602_single_controller/external/state/state_preprint_toml_files.


,source,original_download_URL
0,GOA,https://current.geneontology.org/annotations/g...
1,GO_basic,https://current.geneontology.org/ontology/go-b...
2,WikiPathways,https://data.wikipathways.org/current/gmt/wiki...
3,KEGG_list_hsa,https://rest.kegg.jp/list/hsa
4,KEGG_list_pathway,https://rest.kegg.jp/list/pathway/hsa
5,KEGG_link_pathway,https://rest.kegg.jp/link/pathway/hsa


## Original processing code

These are the original functions used for this analysis. `recovered_pipeline.py` changes their filesystem destinations and uses the original cached downloads; it writes regenerated products only to `computed/`. Set `force=True` in `run_stage` to recompute this stage after its prerequisites.

In [3]:
ns.show_code('controls',10)

/opt/homebrew/anaconda3/envs/fleming/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


```python
def standardized_ols(df: pd.DataFrame, y_col: str, include_de_count: bool) -> dict[str, object]:
    work = df.dropna(subset=[y_col, SUPPORT_COL, "reactome_pathway_count"]).copy()
    work = work[work["reactome_pathway_count"] > 0].copy()
    y = pd.to_numeric(work[y_col], errors="coerce").to_numpy(float)
    cols = []
    names = []

    def add_col(name: str, values: np.ndarray, standardize: bool = True) -> None:
        vals = values.astype(float)
        if standardize:
            sd = float(np.std(vals))
            vals = (vals - float(np.mean(vals))) / sd if sd > 0 else vals * 0
        cols.append(vals)
        names.append(name)

    add_col("support_log_mean_z", pd.to_numeric(work[SUPPORT_COL], errors="coerce").to_numpy(float))
    add_col("reactome_pathway_count_z", pd.to_numeric(work["reactome_pathway_count"], errors="coerce").to_numpy(float))
    if include_de_count:
        add_col("log1p_de_nsig_counts_real_z", np.log1p(pd.to_numeric(work["de_nsig_counts_real"], errors="coerce").fillna(0).to_numpy(float)))
    for col in ["context", "representation", "spectra_subset"]:
        dummies = pd.get_dummies(work[col], prefix=col, drop_first=True, dtype=float)
        for dcol in dummies.columns:
            cols.append(dummies[dcol].to_numpy(float))
            names.append(dcol)
    x = np.column_stack([np.ones(len(work)), *cols])
    beta, residuals, rank, _ = np.linalg.lstsq(x, y, rcond=None)
    yhat = x @ beta
    ss_res = float(np.sum((y - yhat) ** 2))
    ss_tot = float(np.sum((y - np.mean(y)) ** 2))
    r2 = 1.0 - ss_res / ss_tot if ss_tot else float("nan")
    n, p = x.shape
    sigma2 = ss_res / max(n - p, 1)
    cov = sigma2 * np.linalg.pinv(x.T @ x)
    se = np.sqrt(np.diag(cov))
    coefs = [{"term": "intercept", "coef": float(beta[0]), "se": float(se[0])}]
    for name, coef, serr in zip(names, beta[1:], se[1:]):
        coefs.append({"term": name, "coef": float(coef), "se": float(serr), "t": float(coef / serr) if serr else float("nan")})
    return {"n_rows": int(n), "r2": r2, "include_de_count": include_de_count, "coefficients": coefs}

```

## Recomputed sample membership, values and result

In [4]:
display(verification)

{'output': 'continued/artifacts/controls_replogle_best_axis_summary.json'}

In [5]:
ns.display_json(rp.COMPUTED/'continued/artifacts/controls_replogle_best_axis_regression.json')

```json
{
  "without_de_count": {
    "n_rows": 6640,
    "r2": 0.1641922046102613,
    "include_de_count": false,
    "coefficients": [
      {
        "term": "intercept",
        "coef": 0.18203543767509486,
        "se": 0.0029175848006698186
      },
      {
        "term": "support_log_mean_z",
        "coef": 0.07787641538415838,
        "se": 0.0026267829092452536,
        "t": 29.647069466633006
      },
      {
        "term": "reactome_pathway_count_z",
        "coef": 0.046461609741254184,
        "se": 0.0026215587634465768,
        "t": 17.72289463394323
      },
      {
        "term": "context_jurkat",
        "coef": 0.025381120888192797,
        "se": 0.00421025165221888,
        "t": 6.028409459756753
      },
      {
        "term": "context_k562",
        "coef": -0.022007111639300546,
        "se": 0.0074344248236732375,
        "t": -2.960163316094595
      },
      {
        "term": "context_rpe1",
        "coef": 0.10562447401139191,
        "se": 0.004193544188387848,
        "t": 25.187399790342454
      },
      {
        "term": "representation_SE",
        "coef": -0.05356743360124029,
        "se": 0.005104029345038659,
        "t": -10.49512649321074
      },
      {
        "term": "spectra_subset_discovery_hepg2_k562",
        "coef": 0.05102984277550912,
        "se": 0.004413767591095574,
        "t": 11.561515581032808
      }
    ]
  },
  "with_de_count_outcome_control": {
    "n_rows": 6640,
    "r2": 0.5714659339766228,
    "include_de_count": true,
    "coefficients": [
      {
        "term": "intercept",
        "coef": 0.17228577746794985,
        "se": 0.0020928820476762418
      },
      {
        "term": "support_log_mean_z",
        "coef": 0.02844803805288416,
        "se": 0.0019814045841198336,
        "t": 14.357510970189441
      },
      {
        "term": "reactome_pathway_count_z",
        "coef": 0.0149809475289914,
        "se": 0.001918718601972729,
        "t": 7.8077877149826715
      },
      {
        "term": "log1p_de_nsig_counts_real_z",
        "coef": 0.16287927384870018,
        "se": 0.0020517563566334724,
        "t": 79.38529022810143
      },
      {
        "term": "context_jurkat",
        "coef": 0.06571133467454801,
        "se": 0.0030574541232596874,
        "t": 21.492173561868604
      },
      {
        "term": "context_k562",
        "coef": 0.00669274743116927,
        "se": 0.005336034813980758,
        "t": 1.254254828629273
      },
      {
        "term": "context_rpe1",
        "coef": 0.029330554915557024,
        "se": 0.0031530245707637587,
        "t": 9.302355328126184
      },
      {
        "term": "representation_SE",
        "coef": -0.0535674336012406,
        "se": 0.0036549831114163606,
        "t": -14.656000306519177
      },
      {
        "term": "spectra_subset_discovery_hepg2_k562",
        "coef": 0.07724388787784402,
        "se": 0.003177891011970756,
        "t": 24.306651042114105
      }
    ]
  },
  "secondary_de_recall_with_de_count": {
    "n_rows": 6640,
    "r2": 0.48522812055698816,
    "include_de_count": true,
    "coefficients": [
      {
        "term": "intercept",
        "coef": 0.2021942536572313,
        "se": 0.0025618774487350523
      },
      {
        "term": "support_log_mean_z",
        "coef": 0.033558135848480504,
        "se": 0.002425418922443787,
        "t": 13.836016342557526
      },
      {
        "term": "reactome_pathway_count_z",
        "coef": 0.017179008522453854,
        "se": 0.002348685594737725,
        "t": 7.314307441125262
      },
      {
        "term": "log1p_de_nsig_counts_real_z",
        "coef": 0.16517119129450655,
        "se": 0.0025115358728382657,
        "t": 65.76501378331817
      },
      {
        "term": "context_jurkat",
        "coef": 0.06821505631174658,
        "se": 0.003742601155004361,
        "t": 18.226643312107644
      },
      {
        "term": "context_k562",
        "coef": -0.004318185214757199,
        "se": 0.006531790585513764,
        "t": -0.6611028259745637
      },
      {
        "term": "context_rpe1",
        "coef": 0.0358519458574676,
        "se": 0.0038595880508965814,
        "t": 9.289060227331568
      },
      {
        "term": "representation_SE",
        "coef": -0.052623741657814964,
        "se": 0.004474030831809951,
        "t": -11.762042693953953
      },
      {
        "term": "spectra_subset_discovery_hepg2_k562",
        "coef": 0.09812725148801618,
        "se": 0.0038900268302961344,
        "t": 25.225340535902188
      }
    ]
  },
  "secondary_pearson_with_de_count": {
    "n_rows": 6640,
    "r2": 0.6770830342467552,
    "include_de_count": true,
    "coefficients": [
      {
        "term": "intercept",
        "coef": 0.31419921632175823,
        "se": 0.0025933782242446184
      },
      {
        "term": "support_log_mean_z",
        "coef": 0.03458059118390937,
        "se": 0.0024552418076213274,
        "t": 14.084393266914727
      },
      {
        "term": "reactome_pathway_count_z",
        "coef": 0.008745853237189214,
        "se": 0.0023775649690024525,
        "t": 3.6784917977903606
      },
      {
        "term": "log1p_de_nsig_counts_real_z",
        "coef": 0.26026890807382236,
        "se": 0.002542417649698265,
        "t": 102.37063454334938
      },
      {
        "term": "context_jurkat",
        "coef": 0.11697015015431936,
        "se": 0.0037886200771287764,
        "t": 30.874077572583033
      },
      {
        "term": "context_k562",
        "coef": -0.0031135182207248356,
        "se": 0.006612105305100121,
        "t": -0.4708815236689112
      },
      {
        "term": "context_rpe1",
        "coef": 0.06631881126182801,
        "se": 0.003907045440714634,
        "t": 16.974159187075568
      },
      {
        "term": "representation_SE",
        "coef": -0.04885156651477707,
        "se": 0.004529043393369174,
        "t": -10.786288024155182
      },
      {
        "term": "spectra_subset_discovery_hepg2_k562",
        "coef": 0.13091025490560942,
        "se": 0.003937858494518735,
        "t": 33.24402212213281
      }
    ]
  }
}
```

## Original agent decision and supporting diagnostics

The following records preserve the agent’s actual determination, including rejected candidates and controls. Their historical status labels are shown as recorded.

In [6]:
ns.display_json(rp.CONTINUED/'artifacts/controls_replogle_best_axis_regression.json')
ns.display_json(rp.CONTINUED/'artifacts/controls_replogle_best_axis_summary.json')

```json
{
  "without_de_count": {
    "n_rows": 6640,
    "r2": 0.1641922046102613,
    "include_de_count": false,
    "coefficients": [
      {
        "term": "intercept",
        "coef": 0.18203543767509506,
        "se": 0.002917584800669816
      },
      {
        "term": "support_log_mean_z",
        "coef": 0.07787641538415785,
        "se": 0.0026267829092452544,
        "t": 29.647069466632797
      },
      {
        "term": "reactome_pathway_count_z",
        "coef": 0.04646160974125387,
        "se": 0.002621558763446576,
        "t": 17.722894633943117
      },
      {
        "term": "context_jurkat",
        "coef": 0.025381120888193026,
        "se": 0.0042102516522188795,
        "t": 6.0284094597568085
      },
      {
        "term": "context_k562",
        "coef": -0.022007111639300622,
        "se": 0.0074344248236732375,
        "t": -2.9601633160946053
      },
      {
        "term": "context_rpe1",
        "coef": 0.105624474011392,
        "se": 0.004193544188387847,
        "t": 25.18739979034248
      },
      {
        "term": "representation_SE",
        "coef": -0.053567433601240576,
        "se": 0.005104029345038653,
        "t": -10.495126493210806
      },
      {
        "term": "spectra_subset_discovery_hepg2_k562",
        "coef": 0.051029842775509406,
        "se": 0.0044137675910955725,
        "t": 11.561515581032875
      }
    ]
  },
  "with_de_count_outcome_control": {
    "n_rows": 6640,
    "r2": 0.5714659339766228,
    "include_de_count": true,
    "coefficients": [
      {
        "term": "intercept",
        "coef": 0.17228577746794976,
        "se": 0.0020928820476762418
      },
      {
        "term": "support_log_mean_z",
        "coef": 0.02844803805288392,
        "se": 0.001981404584119833,
        "t": 14.357510970189324
      },
      {
        "term": "reactome_pathway_count_z",
        "coef": 0.014980947528991357,
        "se": 0.001918718601972729,
        "t": 7.807787714982649
      },
      {
        "term": "log1p_de_nsig_counts_real_z",
        "coef": 0.16287927384870038,
        "se": 0.002051756356633472,
        "t": 79.38529022810154
      },
      {
        "term": "context_jurkat",
        "coef": 0.06571133467454804,
        "se": 0.0030574541232596874,
        "t": 21.492173561868615
      },
      {
        "term": "context_k562",
        "coef": 0.006692747431169138,
        "se": 0.005336034813980759,
        "t": 1.254254828629248
      },
      {
        "term": "context_rpe1",
        "coef": 0.029330554915557104,
        "se": 0.0031530245707637595,
        "t": 9.302355328126207
      },
      {
        "term": "representation_SE",
        "coef": -0.05356743360124047,
        "se": 0.0036549831114163593,
        "t": -14.656000306519148
      },
      {
        "term": "spectra_subset_discovery_hepg2_k562",
        "coef": 0.07724388787784423,
        "se": 0.0031778910119707555,
        "t": 24.306651042114172
      }
    ]
  },
  "secondary_de_recall_with_de_count": {
    "n_rows": 6640,
    "r2": 0.4852281205569883,
    "include_de_count": true,
    "coefficients": [
      {
        "term": "intercept",
        "coef": 0.20219425365723123,
        "se": 0.0025618774487350523
      },
      {
        "term": "support_log_mean_z",
        "coef": 0.033558135848480275,
        "se": 0.0024254189224437867,
        "t": 13.836016342557434
      },
      {
        "term": "reactome_pathway_count_z",
        "coef": 0.01717900852245385,
        "se": 0.002348685594737725,
        "t": 7.314307441125261
      },
      {
        "term": "log1p_de_nsig_counts_real_z",
        "coef": 0.16517119129450678,
        "se": 0.0025115358728382653,
        "t": 65.76501378331827
      },
      {
        "term": "context_jurkat",
        "coef": 0.06821505631174662,
        "se": 0.003742601155004361,
        "t": 18.226643312107658
      },
      {
        "term": "context_k562",
        "coef": -0.004318185214757282,
        "se": 0.006531790585513764,
        "t": -0.6611028259745763
      },
      {
        "term": "context_rpe1",
        "coef": 0.03585194585746777,
        "se": 0.0038595880508965827,
        "t": 9.289060227331607
      },
      {
        "term": "representation_SE",
        "coef": -0.0526237416578148,
        "se": 0.004474030831809949,
        "t": -11.762042693953921
      },
      {
        "term": "spectra_subset_discovery_hepg2_k562",
        "coef": 0.09812725148801633,
        "se": 0.003890026830296134,
        "t": 25.22534053590223
      }
    ]
  },
  "secondary_pearson_with_de_count": {
    "n_rows": 6640,
    "r2": 0.6770830342467552,
    "include_de_count": true,
    "coefficients": [
      {
        "term": "intercept",
        "coef": 0.31419921632175823,
        "se": 0.002593378224244618
      },
      {
        "term": "support_log_mean_z",
        "coef": 0.034580591183909146,
        "se": 0.0024552418076213265,
        "t": 14.084393266914642
      },
      {
        "term": "reactome_pathway_count_z",
        "coef": 0.008745853237189205,
        "se": 0.002377564969002452,
        "t": 3.6784917977903575
      },
      {
        "term": "log1p_de_nsig_counts_real_z",
        "coef": 0.2602689080738226,
        "se": 0.0025424176496982643,
        "t": 102.3706345433495
      },
      {
        "term": "context_jurkat",
        "coef": 0.11697015015431958,
        "se": 0.003788620077128776,
        "t": 30.874077572583097
      },
      {
        "term": "context_k562",
        "coef": -0.0031135182207249965,
        "se": 0.006612105305100121,
        "t": -0.4708815236689355
      },
      {
        "term": "context_rpe1",
        "coef": 0.06631881126182826,
        "se": 0.0039070454407146345,
        "t": 16.974159187075628
      },
      {
        "term": "representation_SE",
        "coef": -0.04885156651477692,
        "se": 0.004529043393369172,
        "t": -10.786288024155153
      },
      {
        "term": "spectra_subset_discovery_hepg2_k562",
        "coef": 0.13091025490560973,
        "se": 0.003937858494518734,
        "t": 33.2440221221329
      }
    ]
  }
}
```

```json
{
  "status": "completed",
  "interpretation": "The refined Reactome support axis remains a current-pool confirmed boundary. Controls test whether it is explained by annotation presence, context, representation, pathway annotation count, or DE-count severity.",
  "artifacts": [
    "/ewsc/yektefai/spectra_runs/state_generalizability_continued_20260603/artifacts/controls_replogle_best_axis_level_summary_all.csv",
    "/ewsc/yektefai/spectra_runs/state_generalizability_continued_20260603/artifacts/controls_replogle_best_axis_level_summary_no_l0.csv",
    "/ewsc/yektefai/spectra_runs/state_generalizability_continued_20260603/artifacts/controls_replogle_best_axis_high_low_gaps.csv",
    "/ewsc/yektefai/spectra_runs/state_generalizability_continued_20260603/artifacts/controls_replogle_best_axis_pathway_count_matched.csv",
    "/ewsc/yektefai/spectra_runs/state_generalizability_continued_20260603/artifacts/controls_replogle_best_axis_regression.json"
  ]
}
```

All source files, downloaded inputs and historical outputs are checksummed in `original_manifest.json`. The full preliminary candidate screening and download inventory are retained in notebook `00-original-downloads-and-axis-history.ipynb`.